# Machine Learning Lab – Practical 3: Bank Customer Churn Prediction Using ANN

**Algorithm:** Artificial Neural Network (ANN) with Dropout and Early Stopping  
**Dataset:** `Churn_Modelling.csv`  
**Aim:** To build an Artificial Neural Network model that predicts whether a bank customer will leave (churn) the bank.

## What does customer churn mean?
Customer churn occurs when a customer stops using a company's or bank's services. In this practical, the target column is `Exited`:
- `0` = Customer did not leave the bank.
- `1` = Customer left the bank.

**Reference:** This notebook follows your uploaded `NNClassifier.ipynb` and keeps its main sequence: load data, separate features and target, split, scale, build the initial ANN, improve it with Dropout and Early Stopping, train, predict and evaluate. Additional explanations and evaluation visualizations are included to make the practical easier to study.

## 1. Theory and concepts required

### Artificial Neural Network (ANN)
An ANN is a machine-learning model inspired by the connected neurons in the human brain. It consists of layers of artificial neurons. Each neuron receives inputs, performs a weighted calculation and applies an activation function.

### Main layers
- **Input layer:** Receives the features. Its input size must equal the number of columns in `X_train`.
- **Hidden layer:** Learns patterns from the input data.
- **Output layer:** Produces the prediction. Here, one neuron with sigmoid activation outputs a probability between 0 and 1.

### Important terms
- **Epoch:** One complete pass through all training samples.
- **Batch size:** Number of samples processed before the model updates its weights.
- **Learning:** The model adjusts weights to reduce its prediction error.
- **Optimizer:** Algorithm that updates weights. `adam` is an adaptive optimization algorithm.
- **Loss function:** Measures how far predictions are from the correct answers.
- **Binary classification:** Predicting one of two classes, such as churn (1) or no churn (0).
- **Overfitting:** When a model learns the training data too closely and performs poorly on new data.
- **Dropout:** Randomly disables a proportion of neurons during training to help reduce overfitting.
- **Early Stopping:** Stops training when validation performance stops improving.
- **Normalization/standardization:** Puts numerical features on comparable scales. `StandardScaler` transforms values using the training mean and standard deviation.

## 2. Import required libraries

- `pandas`: Loads and handles tabular data.
- `numpy`: Performs numerical operations.
- `train_test_split`: Divides data into training and testing sets.
- `StandardScaler`: Standardizes numerical feature values.
- Evaluation metrics: Measure model performance.
- Keras `Sequential`, `Dense`, `Dropout`, `Input`: Build the neural network.
- `EarlyStopping`: Stops training when validation loss no longer improves.
- `matplotlib`: Displays evaluation graphs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

## 3. Read and inspect the dataset

`pd.read_csv()` loads the CSV file into a DataFrame. `shape` displays the number of rows and columns, while `head()` displays the first five records.

The original dataset commonly contains customer information such as credit score, geography, gender, age, tenure, balance, number of products, credit-card status, active-member status and estimated salary. Your CSV is the authority for the exact columns and values.

In [ ]:
data = pd.read_csv("Churn_Modelling.csv")

print("Dataset Shape:", data.shape)
display(data.head())

print("\\nColumn names:")
print(data.columns.tolist())

print("\\nDataset information:")
data.info()

print("\\nMissing values:")
print(data.isnull().sum())

## 4. Separate features and target

### Step A: Remove unnecessary columns
`RowNumber`, `CustomerId` and `Surname` are identifiers or names, rather than useful numeric predictors for this model. The practical removes them.

### Step B: Convert categorical columns into numbers
A neural network requires numeric inputs. `pd.get_dummies()` performs one-hot encoding:
- A category is represented by a binary column.
- `drop_first=True` removes one category from each encoded group to avoid redundant dummy columns.

For example, Geography may become columns such as `Geography_Germany` and `Geography_Spain`, depending on the categories in the dataset.

### Step C: Create X and y
- `X` contains all input features.
- `y` contains the target column `Exited`.

In [ ]:
# Remove identifier/name columns that are not used as model features.
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Convert categorical columns into numeric dummy columns.
data = pd.get_dummies(
    data,
    columns=["Geography", "Gender"],
    drop_first=True
)

# Separate input features and target output.
X = data.drop("Exited", axis=1)
y = data["Exited"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\\nTarget class distribution:")
print(y.value_counts())
print("\\nTarget class percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

## 5. Split data into training and testing sets

The practical uses an 80:20 split:
- 80% for training the ANN.
- 20% for testing its predictions on unseen data.

`random_state=42` makes the split repeatable. `stratify=y` preserves approximately the same proportion of churn and non-churn customers in both sets. This is useful because customer churn datasets can have unequal class counts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Normalize the data using StandardScaler

Features can have very different numerical ranges. For example, Age, CreditScore and Balance are measured on different scales. Standardization helps neural-network training.

`StandardScaler` transforms each feature approximately as:

\[
z = \frac{x - \mu}{\sigma}
\]

where \(x\) is the original value, \(\mu\) is the mean and \(\sigma\) is the standard deviation.

**Important:** Fit the scaler only on `X_train`. Then use the same fitted scaler to transform `X_test`. This prevents information from the test set influencing the training preprocessing.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training data shape after scaling:", X_train.shape)
print("Testing data shape after scaling:", X_test.shape)

## 7. Build the initial Artificial Neural Network

Your original practical builds an initial ANN with:
1. Input layer: number of neurons/input values equals the number of features.
2. First hidden layer: 64 neurons, ReLU activation.
3. Second hidden layer: 32 neurons, ReLU activation.
4. Output layer: 1 neuron, sigmoid activation.

### Activation functions
- **ReLU:** Returns zero for negative input and the input itself for positive input. It is commonly used in hidden layers.
- **Sigmoid:** Converts the output into a value between 0 and 1, suitable for binary classification.

### Compile settings
- `optimizer="adam"`: updates model weights during training.
- `loss="binary_crossentropy"`: suitable loss function for binary classification.
- `metrics=["accuracy"]`: displays the fraction of correct predictions during training/evaluation.

`model.summary()` displays the layer structure and number of trainable parameters.

**Note:** In your reference code, this initial model is created and summarized, but it is not trained. The improved model in the next section is the one that is trained and evaluated.

In [ ]:
initial_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

initial_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

initial_model.summary()

## 8. Improve the ANN using Dropout and Early Stopping

Your improved network adds Dropout layers and an extra hidden layer.

### Architecture
- Input layer.
- Dense layer with 64 neurons and ReLU.
- Dropout of 0.3: randomly disables 30% of the layer's outputs during training.
- Dense layer with 32 neurons and ReLU.
- Dropout of 0.2: randomly disables 20% of the layer's outputs during training.
- Dense layer with 16 neurons and ReLU.
- Output layer with 1 neuron and sigmoid.

### Early Stopping
The callback monitors `val_loss` (validation loss).
- `patience=5`: training can continue for up to five epochs without validation-loss improvement before stopping.
- `restore_best_weights=True`: restores the model weights from the epoch with the best validation loss.

Dropout and Early Stopping are regularization techniques that can help reduce overfitting. They do not guarantee that overfitting will be eliminated.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model.summary()

## 9. Train the improved model

`model.fit()` trains the network using the training data.

- `epochs=50`: maximum number of complete training passes.
- `batch_size=32`: processes 32 samples per batch before updating weights.
- `validation_split=0.2`: reserves 20% of the training data for validation.
- `callbacks=[early_stop]`: uses the Early Stopping callback.
- `verbose=1`: displays training progress.

The returned `history` object stores loss and accuracy values for each epoch.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## 10. Visualize training and validation performance

The first graph compares training loss with validation loss. The second compares training accuracy with validation accuracy.

- Loss generally should decrease as the model learns.
- Accuracy generally should increase.
- A large gap between training and validation performance can indicate overfitting.
- Early stopping may finish training before all 50 epochs if validation loss stops improving.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Predict the test data

`model.predict(X_test)` produces a probability for each test customer. Because the output layer uses sigmoid, each probability is between 0 and 1.

The practical uses a threshold of 0.5:
- Probability greater than 0.5 → class `1` (customer predicted to churn).
- Probability less than or equal to 0.5 → class `0` (customer predicted not to churn).

`.astype(int)` converts Boolean results to integer labels, and `.ravel()` makes the predictions a one-dimensional array for metric functions.

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("First 10 predicted probabilities:")
print(y_pred_prob[:10].ravel())

print("\\nFirst 10 predicted classes:")
print(y_pred[:10])

## 12. Evaluate the model

### Accuracy
\[
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
\]

Accuracy is the proportion of all predictions that are correct.

### Precision
\[
Precision = \frac{TP}{TP + FP}
\]

Of the customers predicted to churn, precision tells us how many actually churned.

### Recall
\[
Recall = \frac{TP}{TP + FN}
\]

Of the customers who actually churned, recall tells us how many the model correctly identified.

### F1-score
\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

F1-score balances precision and recall.

**Why include more than accuracy?** Churn datasets may have more non-churn customers than churn customers. A model can achieve seemingly high accuracy while missing many actual churn cases. Precision, recall and F1-score provide a fuller picture.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("----- ANN MODEL PERFORMANCE -----")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Exited (0)", "Exited (1)"],
    zero_division=0
))

## 13. Confusion matrix

A confusion matrix compares actual labels with predicted labels. For binary classification, scikit-learn displays it in this order:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | True Negative (TN) | False Positive (FP) |
| Actual 1 | False Negative (FN) | True Positive (TP) |

- **TN:** Correctly predicted a customer would stay.
- **FP:** Predicted churn, but the customer stayed.
- **FN:** Predicted the customer would stay, but the customer churned.
- **TP:** Correctly predicted a customer would churn.

For churn prediction, False Negatives can be important because the bank may fail to identify a customer who is about to leave.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Exited (0)", "Exited (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("ANN Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Conclusion

In this practical, a bank customer churn prediction model is developed using an Artificial Neural Network. Categorical variables are converted into numerical dummy variables, the dataset is split into training and testing sets, and features are standardized. An initial ANN is constructed, then an improved ANN is created using Dropout and Early Stopping. The improved model is trained and evaluated using accuracy, precision, recall, F1-score and a confusion matrix.

**Result statement:** After running the notebook, write the actual metrics printed by your execution and briefly state how well the model identifies customers who churn. Do not use assumed accuracy values.

## 15. Important viva questions and answers

1. **What is customer churn?**  
   Customer churn means a customer stops using the services of a company or bank.

2. **What is the aim of this practical?**  
   To predict whether a bank customer will leave the bank using an ANN model.

3. **What is ANN?**  
   An Artificial Neural Network is a machine-learning model made of connected layers of artificial neurons that learn patterns from data.

4. **Which column is the target variable?**  
   `Exited`.

5. **What do 0 and 1 represent in the target?**  
   0 means the customer did not exit; 1 means the customer exited.

6. **Why are RowNumber, CustomerId and Surname removed?**  
   They are identifiers or names and are not used as predictive features in this practical.

7. **Why is one-hot encoding used?**  
   To convert categorical values such as Geography and Gender into numerical columns that the ANN can process.

8. **What is `pd.get_dummies()`?**  
   A Pandas function that converts categorical columns into dummy/indicator columns.

9. **Why is the dataset split into training and testing sets?**  
   The model learns from training data and is evaluated on separate unseen test data.

10. **What is stratified sampling?**  
    It keeps approximately the same class proportions in training and testing sets.

11. **Why is StandardScaler used?**  
    It standardizes features so that they have comparable scales, helping neural-network training.

12. **Why is the scaler fitted only on training data?**  
    To prevent information from the test data leaking into the training process.

13. **What is a neuron?**  
    A basic processing unit in a neural network that combines inputs, weights and a bias, then applies an activation function.

14. **What is a Dense layer?**  
    A fully connected layer in which each neuron receives input from all neurons in the previous layer.

15. **What is ReLU?**  
    An activation function that outputs zero for negative inputs and the input itself for positive inputs.

16. **What is sigmoid activation?**  
    An activation function that maps a value to a range between 0 and 1, often used for binary classification output.

17. **Why is the output layer made of one neuron?**  
    Because the task has two classes and one sigmoid output represents the probability of class 1.

18. **What is binary cross-entropy?**  
    A loss function used to measure prediction error in binary classification.

19. **What is the Adam optimizer?**  
    An optimization algorithm that adapts learning rates while updating model weights.

20. **What is an epoch?**  
    One complete pass through the training dataset.

21. **What is batch size?**  
    The number of training samples processed before the model updates its weights.

22. **What is overfitting?**  
    When a model learns training data too closely and does not generalize well to unseen data.

23. **What is Dropout?**  
    A regularization technique that randomly disables a fraction of neurons during training to help reduce overfitting.

24. **What does Dropout(0.3) mean?**  
    It randomly disables 30% of the relevant layer's outputs during each training step.

25. **What is Early Stopping?**  
    A callback that stops training when the monitored validation metric stops improving.

26. **What does patience=5 mean?**  
    Training waits for five epochs without improvement before stopping.

27. **What does restore_best_weights=True do?**  
    It restores the model weights from the epoch with the best monitored validation loss.

28. **What is a confusion matrix?**  
    A table that compares actual and predicted class labels.

29. **What is the difference between precision and recall?**  
    Precision measures correctness among predicted positives; recall measures how many actual positives were found.

30. **Why is F1-score useful?**  
    It combines precision and recall, which is useful when class counts are unequal.

31. **What is the difference between validation data and test data?**  
    Validation data helps monitor/tune training; test data is held aside for final evaluation.

32. **Does high accuracy always mean a good churn model?**  
    No. If classes are imbalanced, accuracy can be high even when the model misses many churn customers. Recall and F1-score should also be checked.

## 16. Execution checklist and troubleshooting

1. Keep `Churn_Modelling.csv` in the same folder as this notebook, or update the path in the read cell.
2. Install the required packages if they are missing:
   - `pandas`
   - `numpy`
   - `scikit-learn`
   - `matplotlib`
   - `tensorflow`
3. Run all cells from top to bottom in order.
4. If `FileNotFoundError` appears, check the CSV filename and notebook working directory.
5. If a column-name `KeyError` appears, inspect `data.columns.tolist()` and compare the names with the code.
6. If TensorFlow cannot be imported, install a TensorFlow version compatible with your Python environment and restart the notebook kernel.
7. Training results and scores depend on the actual CSV and software environment.

**Execution note:** This improved notebook has been prepared from your uploaded `NNClassifier.ipynb`. It has not been executed against your local `Churn_Modelling.csv`, so no accuracy or confusion-matrix values are assumed here.